# XLAV — E2: text prompt và visual prompt (CLIPSeg, PhraseCut miniv)

Câu hỏi: chỉ cho CLIPSeg biết vật cần tìm **bằng chữ** hay **bằng một ảnh mẫu (support)** thì tốt hơn,
và ảnh mẫu nên được xử lý thế nào? Mỗi query miniv được dự đoán với 5 biến thể prompt:

| Biến thể | Prompt |
|---|---|
| `text_phrase` | Cụm từ gốc, ví dụ `red truck` (giống E1, là gốc để so sánh) |
| `text_name` | Chỉ danh từ chính `truck`, cùng mức thông tin với ảnh support |
| `visual_crop` | Ảnh support crop theo bbox vật (nới 10%) |
| `visual_dark` | Cả ảnh support, nền ngoài vật giữ 10% độ sáng |
| `visual_crop_dark` | Kết hợp crop và tối nền |

**Support** lấy từ **ảnh miniv khác** có cùng danh từ chính (`phrase_structure.name`). Mask dùng để crop/tối nền
là GT **của ảnh support**; mask GT của ảnh query không bao giờ được dùng để tạo prompt. Mỗi query dùng đúng
`SUPPORTS_PER_QUERY` support từ các ảnh khác nhau; metric của biến thể visual là trung bình trên các support.
Danh sách cặp lưu ở `e2_pairs.json`.

1. Bật **GPU T4 x2** và **Internet** trong Settings của notebook Kaggle.
2. **Run All**. Kết quả ở `/kaggle/working/clipseg_e2_text_vs_visual`, kèm file ZIP.

Đây là phiên bản rút gọn ý tưởng Bảng 2 của paper (paper đo ΔP trên LVIS, có thêm blur); số liệu không so trực tiếp.


## 1. Dependency

Giữ PyTorch/CUDA, NumPy, Pillow, scikit-learn và matplotlib sẵn có của Kaggle.
Cell này chỉ pin Transformers và gdown; không cài wheel PyTorch CPU.
Nếu kernel đã import Transformers phiên bản khác trước đó, restart session rồi chạy lại từ đầu.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.55.4", "gdown==5.2.0",
])


## 2. Cấu hình

`MAX_QUERIES=30` theo kế hoạch; nếu không đủ query có support hợp lệ, notebook chạy với số có được và ghi rõ N.
`MIN_SUPPORT_BOX_FRACTION` loại support quá nhỏ (tổng diện tích box của vật / diện tích ảnh support),
vì crop từ vật vài pixel không còn là ảnh mẫu có nghĩa.


In [ ]:
from pathlib import Path

MAX_QUERIES = 30                 # kế hoạch E2: 30 query; 8 = smoke
SUPPORTS_PER_QUERY = 3           # mỗi query dùng đúng K support từ K ảnh khác nhau
MIN_SUPPORT_BOX_FRACTION = 0.02  # bỏ support có vật quá nhỏ
CROP_PADDING = 0.1               # nới bbox support 10% mỗi phía trước khi crop
DARK_FACTOR = 0.1                # nền giữ 10% độ sáng, như Bảng 2 của paper
OUTPUT_NAME = "clipseg_e2_text_vs_visual"

DATA_ROOT = None                 # ví dụ: "/kaggle/input/phrasecut-miniv/VGPhraseCut_v0"
MODEL_PATH = None                # thư mục checkpoint offline, hoặc None
WORK_ROOT = Path("/kaggle/working")
REQUIRE_GPU = True               # False chỉ khi kiểm tra trên máy CPU
USE_AMP = False                  # mặc định FP32 như E1
SEED = 42
THRESHOLD = 0.3
CALIBRATION_FRACTION = 0.2
THRESHOLD_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
N_BOOTSTRAP = 2000               # bootstrap theo ảnh cho CI 95% của ΔIoU
TIE_TOLERANCE = 0.01             # |ΔIoU| ≤ 0,01 tính là hòa
SAVE_MASKS = True
N_VISUALIZATIONS = 6
DOWNLOAD_WORKERS = 4

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
PHRASECUT_API_COMMIT = "276470b7773d8a230aee5b5536876fbc485fb04b"


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext

import gdown
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor
from IPython.display import display

assert 0 < CALIBRATION_FRACTION < 1
assert 0 <= THRESHOLD <= 1 and all(0 <= t <= 1 for t in THRESHOLD_GRID)
assert N_BOOTSTRAP >= 100 and TIE_TOLERANCE >= 0
random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA. Bật GPU trong Kaggle Settings rồi restart session.")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        torch.ones(1, device=device).add_(1).item()
    except RuntimeError as exc:
        raise RuntimeError("PyTorch của Kaggle không chạy được trên GPU này (thường gặp với P100). "
                           "Chọn Accelerator GPU T4 x2 rồi restart session.") from exc
assert not USE_AMP or device.type == "cuda", "FP16 AMP yêu cầu CUDA."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT = WORK_ROOT / OUTPUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
# Xóa kết quả lượt cũ để không trộn artifact vào ZIP.
for folder in ["masks", "visualizations"]:
    shutil.rmtree(OUT / folder, ignore_errors=True)
for filename in ["summary.json", "run.json"]:
    (OUT / filename).unlink(missing_ok=True)
(WORK_ROOT / f"{OUTPUT_NAME}.zip").unlink(missing_ok=True)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

versions = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "numpy", "Pillow", "scikit-learn", "matplotlib", "gdown"]}
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
print(json.dumps(versions, indent=2))


## 3. Lấy annotation chính thức và ghép query với support

Annotation tải như E1. Query hợp lệ khi có ít nhất `SUPPORTS_PER_QUERY` **ảnh khác** chứa task cùng danh từ chính
và đủ lớn. Chọn query và support bằng seed 42, rồi chia calibration/evaluation theo image_id của query.
Ảnh cần tải gồm ảnh query và ảnh support.


In [ ]:
def metadata_path(root):
    for name in ["image_data_split.json", "image_data_split3000.json"]:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return root / "image_data_split.json"

if DATA_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob("refer_miniv.json")) if Path("/kaggle/input").exists() else []
    candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
    if len(candidates) > 1:
        raise ValueError("Có nhiều dataset miniv; đặt DATA_ROOT rõ ràng: " + str(candidates))
    data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_miniv"
else:
    data_root = Path(DATA_ROOT)
    if not data_root.is_dir():
        raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
data_root.mkdir(parents=True, exist_ok=True)
annotation_path = data_root / "refer_miniv.json"
info_path = metadata_path(data_root)

def ensure_json(path, file_id):
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError(f"JSON không phải danh sách annotation: {path}")
        return value
    if str(path).startswith("/kaggle/input/"):
        raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
    temp = path.with_suffix(".download")
    try:
        result = gdown.download(id=file_id, output=str(temp), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive không trả về file.")
        value = json.loads(temp.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError("Nội dung download không phải annotation JSON.")
        temp.replace(path)
        return value
    except Exception as exc:
        raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc

tasks = ensure_json(annotation_path, "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E")
image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == "miniv"}
assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
assert all(int(t["image_id"]) in info_by_id for t in tasks), "Annotation không thuộc split miniv."
assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."

assert all(str(t.get("phrase_structure", {}).get("name", "")).strip() for t in tasks), "Thiếu phrase_structure.name."
assert SUPPORTS_PER_QUERY >= 1 and MAX_QUERIES >= 1 and 0 <= MIN_SUPPORT_BOX_FRACTION < 1

def target_key(task):
    return task["phrase_structure"]["name"].strip().lower()

def box_fraction(task):
    info = info_by_id[int(task["image_id"])]
    area = sum(max(w, 0) * max(h, 0) for _, _, w, h in task["instance_boxes"])
    return area / (int(info["width"]) * int(info["height"]))

ordered = sorted(tasks, key=lambda t: str(t["task_id"]))
supports_by_key = defaultdict(dict)   # key -> image_id -> một task support (task_id nhỏ nhất)
for task in ordered:
    if box_fraction(task) >= MIN_SUPPORT_BOX_FRACTION:
        supports_by_key[target_key(task)].setdefault(int(task["image_id"]), task)
eligible = []
for task in ordered:
    candidates = [s for image_id, s in sorted(supports_by_key[target_key(task)].items())
                  if image_id != int(task["image_id"])]
    if len(candidates) >= SUPPORTS_PER_QUERY:
        eligible.append((task, candidates))
rng = random.Random(SEED)
chosen = rng.sample(eligible, min(MAX_QUERIES, len(eligible)))
chosen.sort(key=lambda pair: (int(pair[0]["image_id"]), str(pair[0]["task_id"])))
pairs = [{"query": task, "supports": rng.sample(candidates, SUPPORTS_PER_QUERY)} for task, candidates in chosen]
if len(pairs) < MAX_QUERIES:
    print(f"Chỉ có {len(pairs)} query có đủ support hợp lệ (yêu cầu {MAX_QUERIES}); báo cáo đúng N này.")
if len(pairs) < 2:
    raise ValueError("Không đủ query có support hợp lệ; giảm SUPPORTS_PER_QUERY hoặc MIN_SUPPORT_BOX_FRACTION.")
for pair in pairs:
    assert all(int(s["image_id"]) != int(pair["query"]["image_id"]) for s in pair["supports"]), "Support trùng ảnh query."
    assert len({int(s["image_id"]) for s in pair["supports"]}) == SUPPORTS_PER_QUERY

selected = [pair["query"] for pair in pairs]
query_image_ids = sorted({int(t["image_id"]) for t in selected})
if len(query_image_ids) < 2:
    raise ValueError("Cần ít nhất 2 ảnh query để chia calibration/evaluation; tăng MAX_QUERIES.")
shuffled_ids = query_image_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)
n_cal = min(len(query_image_ids) - 1, max(1, round(len(query_image_ids) * CALIBRATION_FRACTION)))
calibration_ids = set(shuffled_ids[:n_cal])
group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in query_image_ids}
image_ids = sorted(set(query_image_ids) | {int(s["image_id"]) for p in pairs for s in p["supports"]})
write_json(OUT / "e2_pairs.json", [{"task_id": str(p["query"]["task_id"]), "image_id": int(p["query"]["image_id"]),
    "phrase": p["query"]["phrase"], "target_key": target_key(p["query"]),
    "group": group_for_image[int(p["query"]["image_id"])],
    "supports": [{"task_id": str(s["task_id"]), "image_id": int(s["image_id"]), "phrase": s["phrase"],
                  "box_fraction": round(box_fraction(s), 4)} for s in p["supports"]]} for p in pairs])
print(f"Annotation: {len(tasks)} queries / {len(info_by_id)} miniv images; eligible queries: {len(eligible)}")
print(f"Selected: {len(pairs)} queries / {len(query_image_ids)} query images; calibration {n_cal} images")
print(f"Images to load (query + support): {len(image_ids)}")


## 4. Tải và kiểm tra ảnh

Ưu tiên ảnh đã có trong dataset input. Ảnh thiếu sẽ tải từ URL Visual Genome trong metadata
bằng HTTPS vào working cache, retry có giới hạn. Lỗi tải hoặc kích thước khác metadata làm
dừng lượt chạy: không âm thầm bỏ query để tạo số liệu đẹp hơn.


In [ ]:
image_cache = WORK_ROOT / "phrasecut_image_cache"
image_cache.mkdir(parents=True, exist_ok=True)
local_images = {}
for suffix in ["*.jpg", "*.jpeg", "*.png"]:
    for path in sorted(data_root.rglob(suffix)):
        if path.stem.isdigit() and int(path.stem) in image_ids:
            local_images.setdefault(int(path.stem), path)

def valid_image(path, info):
    with Image.open(path) as image:
        image.load()
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")

def prepare_image(image_id):
    info = info_by_id[image_id]
    path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
    if path.is_file():
        valid_image(path, info)
        return image_id, path
    url = info["url"].replace("http://", "https://", 1)
    temp = path.with_suffix(".download")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temp.open("wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            valid_image(temp, info)
            temp.replace(path)
            return image_id, path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

image_paths, image_errors = {}, []
def prepare_safe(image_id):
    try:
        return prepare_image(image_id), None
    except Exception as exc:
        return None, {"image_id": image_id, "error": str(exc)}

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
        if error:
            image_errors.append(error)
        else:
            image_paths[result[0]] = result[1]
write_json(OUT / "image_errors.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
print("Validated images:", len(image_paths))


## 5. Nạp model, inference và metric

Model, rasterize GT và metric giống hệt E1. `predict` nhận **hoặc** danh sách text prompt
**hoặc** danh sách ảnh support (truyền vào `conditional_pixel_values` của CLIPSeg),
luôn dự đoán trên cùng một ảnh query. Logits nội suy bilinear về kích thước gốc rồi sigmoid.

`paired_comparison` so từng query với biến thể gốc: ΔIoU = biến thể − gốc, CI 95% bằng
bootstrap theo **ảnh** (các query cùng ảnh không độc lập), kèm số query thắng/hòa/thua.


In [ ]:
load_kwargs = {"local_files_only": True} if MODEL_PATH else {"revision": MODEL_REVISION}
model_source = str(MODEL_PATH) if MODEL_PATH else MODEL_ID
processor = CLIPSegProcessor.from_pretrained(model_source, use_fast=False, **load_kwargs)
model = CLIPSegForImageSegmentation.from_pretrained(model_source, **load_kwargs).to(device).eval()
assert processor.image_processor.size == {"height": 352, "width": 352}, "Processor không có input 352 như baseline."
assert model.config.vision_config.patch_size == 16, "Baseline yêu cầu ViT-B/16."
print("Model:", model_source, "| parameters:", sum(p.numel() for p in model.parameters()))

def rasterize_gt(task, width, height):
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)
    for instance in task["Polygons"]:
        for polygon in instance:
            if len(polygon) >= 2:
                draw.polygon([(int(x), int(y)) for x, y in polygon], outline=1, fill=1)
    target = np.asarray(mask, dtype=bool)
    if not target.any():
        raise ValueError(f"GT rỗng: task {task['task_id']}")
    return target

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def binary_metrics(probability, target, threshold):
    if probability.shape != target.shape or not target.any():
        raise ValueError("Shape không khớp hoặc GT rỗng.")
    prediction = probability >= threshold
    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    return {"intersection": intersection, "union": union, "iou": intersection / union,
            "dice": 2 * intersection / int(prediction.sum() + target.sum())}

def summarize(rows):
    if not rows:
        raise ValueError("Không có query để tính metric.")
    result = {"n_queries": len(rows), "n_images": len({r["image_id"] for r in rows}),
        "mean_sample_iou": float(np.mean([r["iou"] for r in rows])),
        "cumulative_iou": sum(r["intersection"] for r in rows) / sum(r["union"] for r in rows),
        "mean_dice": float(np.mean([r["dice"] for r in rows]))}
    for cutoff in [0.3, 0.5, 0.7, 0.9]:
        result[f"precision_at_{cutoff}"] = float(np.mean([r["iou"] >= cutoff for r in rows]))
    if "pixel_ap" in rows[0]:
        result["mean_sample_pixel_ap"] = float(np.mean([r["pixel_ap"] for r in rows]))
    return result

@torch.inference_mode()
def predict(image, prompts=None, supports=None):
    """Return [N,H,W] probabilities for N text prompts or N support images on one query image."""
    if (prompts is None) == (supports is None):
        raise ValueError("Truyền text prompts hoặc support images, không cả hai.")
    count = len(prompts) if prompts is not None else len(supports)
    inputs = dict(processor(images=[image] * count, return_tensors="pt"))
    if prompts is not None:
        inputs.update(processor.tokenizer(text=list(prompts), padding=True, truncation=True, return_tensors="pt"))
    else:
        inputs["conditional_pixel_values"] = processor(images=list(supports), return_tensors="pt")["pixel_values"]
    inputs = {k: v.to(device) for k, v in inputs.items()}
    synchronize()
    started = time.perf_counter()
    context = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with context:
        logits = model(**inputs).logits
    if logits.ndim == 2:
        logits = logits.unsqueeze(0)
    probabilities = F.interpolate(logits.float().unsqueeze(1), size=(image.height, image.width),
        mode="bilinear", align_corners=False).squeeze(1).sigmoid()
    synchronize()
    seconds = time.perf_counter() - started
    values = probabilities.cpu().numpy()
    if not np.isfinite(values).all():
        raise ValueError("Prediction chứa NaN/Inf; thử USE_AMP=False.")
    return values, seconds

def evaluate(probability, target, base):
    """Metric tại THRESHOLD (kèm pixel AP) và các dòng quét ngưỡng cho calibration."""
    row = {**base, "threshold": THRESHOLD, **binary_metrics(probability, target, THRESHOLD),
           "pixel_ap": float(average_precision_score(target.ravel(), probability.ravel()))}
    sweep = [{**base, "threshold": t, **binary_metrics(probability, target, t)} for t in THRESHOLD_GRID]
    return row, sweep

def variant_report(rows, sweep_rows):
    """Ngưỡng cố định trên mọi query; ngưỡng bổ sung chọn chỉ trên calibration."""
    table = []
    for threshold in THRESHOLD_GRID:
        subset = [r for r in sweep_rows if r["group"] == "calibration" and r["threshold"] == threshold]
        table.append({"threshold": threshold, **summarize(subset)})
    best = max(table, key=lambda r: (r["mean_sample_iou"], -r["threshold"]))["threshold"]
    return {
        "all_fixed_threshold": summarize(rows),
        "calibration_fixed_threshold": summarize([r for r in rows if r["group"] == "calibration"]),
        "evaluation_fixed_threshold": summarize([r for r in rows if r["group"] == "evaluation"]),
        "calibration_selected_threshold": best,
        "evaluation_calibrated_threshold": summarize([r for r in sweep_rows
            if r["group"] == "evaluation" and r["threshold"] == best]),
    }, table

def paired_comparison(base_iou, other_iou, image_of, n_bootstrap, tie_tolerance, seed):
    """ΔIoU theo query (other − base), CI 95% bootstrap theo ảnh, số query thắng/hòa/thua."""
    query_ids = sorted(base_iou)
    if set(query_ids) != set(other_iou):
        raise ValueError("Hai biến thể phải có cùng tập query.")
    deltas = np.array([other_iou[q] - base_iou[q] for q in query_ids])
    groups = defaultdict(list)
    for index, query_id in enumerate(query_ids):
        groups[image_of[query_id]].append(index)
    clusters = [np.array(indices) for _, indices in sorted(groups.items())]
    rng = np.random.default_rng(seed)
    boot = [deltas[np.concatenate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])].mean()
            for _ in range(n_bootstrap)]
    return {"n_queries": len(deltas), "delta_mean_iou": float(deltas.mean()),
            "ci95_low": float(np.percentile(boot, 2.5)), "ci95_high": float(np.percentile(boot, 97.5)),
            "wins": int((deltas > tie_tolerance).sum()), "ties": int((np.abs(deltas) <= tie_tolerance).sum()),
            "losses": int((deltas < -tie_tolerance).sum())}

def overlay(image, probability):
    rgb = np.asarray(image).astype(np.float32).copy()
    mask = probability >= THRESHOLD
    rgb[mask] = 0.55 * rgb[mask] + 0.45 * np.array([255, 60, 60])
    return rgb.astype(np.uint8)

def load_rgb(image_id):
    with Image.open(image_paths[image_id]) as raw:
        return raw.convert("RGB")

def save_mask(probability, variant, stem):
    if SAVE_MASKS:
        folder = OUT / "masks" / variant
        folder.mkdir(parents=True, exist_ok=True)
        Image.fromarray((probability >= THRESHOLD).astype(np.uint8) * 255).save(folder / f"{stem}.png")

def table_row(name, report, comparison):
    fixed = report["all_fixed_threshold"]
    return {"variant": name, "rows": fixed["n_queries"], "mean_iou": fixed["mean_sample_iou"],
        "cumulative_iou": fixed["cumulative_iou"], "mean_dice": fixed["mean_dice"],
        "mean_pixel_ap": fixed["mean_sample_pixel_ap"], "precision_at_0.5": fixed["precision_at_0.5"],
        "calibrated_threshold": report["calibration_selected_threshold"],
        "evaluation_iou_fixed": report["evaluation_fixed_threshold"]["mean_sample_iou"],
        "evaluation_iou_calibrated": report["evaluation_calibrated_threshold"]["mean_sample_iou"],
        **{key: (comparison or {}).get(key) for key in
           ["delta_mean_iou", "ci95_low", "ci95_high", "wins", "ties", "losses"]}}

def print_table(rows):
    columns = ["variant", "rows", "mean_iou", "cumulative_iou", "mean_pixel_ap", "delta_mean_iou", "ci95_low", "ci95_high", "wins", "losses"]
    print(" | ".join(columns))
    for row in rows:
        print(" | ".join(f"{row[c]:.4f}" if isinstance(row[c], float) else str(row[c]) for c in columns))

def run_metadata(extra):
    inference_seconds = sum(b["forward_resize_seconds"] for b in batch_times)
    prompts = sum(b["prompts"] for b in batch_times)
    return {**extra,
        "model_id": MODEL_ID, "requested_revision": MODEL_REVISION if not MODEL_PATH else None,
        "model_source": model_source, "resolved_commit": getattr(model.config, "_commit_hash", None),
        "phrasecut_api_commit": PHRASECUT_API_COMMIT,
        "annotation_sha256": sha256(annotation_path), "image_metadata_sha256": sha256(info_path),
        "seed": SEED, "threshold": THRESHOLD, "threshold_grid": THRESHOLD_GRID,
        "calibration_fraction": CALIBRATION_FRACTION, "n_bootstrap": N_BOOTSTRAP, "tie_tolerance": TIE_TOLERANCE,
        "precision": "fp16_autocast" if USE_AMP else "fp32", "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "torch_cuda_version": torch.version.cuda, "python": platform.python_version(), "versions": versions,
        "image_processor": processor.image_processor.to_dict(),
        "forward_resize_seconds_total": inference_seconds,
        "forward_resize_ms_per_prompt": 1000 * inference_seconds / prompts,
        "wall_seconds_inference_metrics_saving": elapsed_all,
        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else None,
        "timing_note": "Two warmup batches; forward+resize+sigmoid timed with CUDA sync; excludes preprocessing and metric/save.",
    }


## 6. Tạo visual prompt và chạy 5 biến thể

`support_views` tạo 3 visual prompt từ ảnh support và mask GT **của ảnh support**.
Text: một forward cho `text_phrase` và `text_name`. Visual: mỗi support một forward cho 3 biến thể.
Mask lưu tại `masks/<variant>/<task_id>.png` (text) và `masks/<variant>/<task_id>__<support_task_id>.png` (visual).


In [ ]:
TEXT_VARIANTS = ["text_phrase", "text_name"]
VISUAL_VARIANTS = ["visual_crop", "visual_dark", "visual_crop_dark"]
VARIANTS = TEXT_VARIANTS + VISUAL_VARIANTS
BASELINE_VARIANT = "text_phrase"
(OUT / "visualizations").mkdir(exist_ok=True)

def support_views(image, mask, crop_padding, dark_factor):
    """Crop / tối nền / kết hợp, chỉ dùng mask của chính ảnh support."""
    ys, xs = np.nonzero(mask)
    x0, x1, y0, y1 = int(xs.min()), int(xs.max()) + 1, int(ys.min()), int(ys.max()) + 1
    pad_x, pad_y = round((x1 - x0) * crop_padding), round((y1 - y0) * crop_padding)
    box = (max(0, x0 - pad_x), max(0, y0 - pad_y), min(image.width, x1 + pad_x), min(image.height, y1 + pad_y))
    rgb = np.asarray(image, dtype=np.float32)
    dark = Image.fromarray(np.where(mask[..., None], rgb, rgb * dark_factor).round().astype(np.uint8))
    return {"visual_crop": image.crop(box), "visual_dark": dark, "visual_crop_dark": dark.crop(box)}

views_cache = {}
def views_for(support):
    key = str(support["task_id"])
    if key not in views_cache:
        image = load_rgb(int(support["image_id"]))
        mask = rasterize_gt(support, image.width, image.height)
        views_cache[key] = support_views(image, mask, CROP_PADDING, DARK_FACTOR)
    return views_cache[key]

def save_pair_figure(image, target, pair, probabilities, ious, index):
    task, support = pair["query"], pair["supports"][0]
    panels = [("Ảnh query", image), ("Ground truth", target),
              (f"text_phrase IoU={ious['text_phrase']:.3f}", overlay(image, probabilities["text_phrase"])),
              (f"Support: {support['phrase']}", views_for(support)["visual_crop_dark"])]
    panels += [(f"{v} IoU={ious[v]:.3f}", overlay(image, probabilities[v])) for v in VISUAL_VARIANTS]
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    for ax, (title, value) in zip(axes, panels):
        ax.imshow(value, cmap="gray" if title == "Ground truth" else None)
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{task['task_id']} — {task['phrase']} (visual dùng support đầu tiên)")
    fig.tight_layout()
    fig.savefig(OUT / "visualizations" / f"{index:02d}_{task['task_id']}.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

warmup_image = load_rgb(int(pairs[0]["query"]["image_id"]))
for _ in range(2):
    predict(warmup_image, prompts=[pairs[0]["query"]["phrase"], target_key(pairs[0]["query"])])
    predict(warmup_image, supports=[views_for(pairs[0]["supports"][0])[v] for v in VISUAL_VARIANTS])
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

rows, sweep_rows, batch_times = [], [], []
started_all = time.perf_counter()
for index, pair in enumerate(tqdm(pairs, desc="E2 queries")):
    task = pair["query"]
    task_id, image_id = str(task["task_id"]), int(task["image_id"])
    image = load_rgb(image_id)
    target = rasterize_gt(task, image.width, image.height)
    base = {"task_id": task_id, "image_id": image_id, "phrase": task["phrase"], "target_key": target_key(task),
            "group": group_for_image[image_id], "width": image.width, "height": image.height}
    first_probabilities, first_ious = {}, {}
    text_prompts = {"text_phrase": task["phrase"], "text_name": target_key(task)}
    probabilities, seconds = predict(image, prompts=list(text_prompts.values()))
    batch_times.append({"prompts": len(text_prompts), "forward_resize_seconds": seconds})
    for (variant, prompt), probability in zip(text_prompts.items(), probabilities):
        row, sweep = evaluate(probability, target, {**base, "variant": variant, "prompt": prompt,
                                                    "support_task_id": "", "support_image_id": ""})
        rows.append(row)
        sweep_rows.extend(sweep)
        save_mask(probability, variant, task_id)
        first_probabilities[variant], first_ious[variant] = probability, row["iou"]
    for support_index, support in enumerate(pair["supports"]):
        views = views_for(support)
        probabilities, seconds = predict(image, supports=[views[v] for v in VISUAL_VARIANTS])
        batch_times.append({"prompts": len(VISUAL_VARIANTS), "forward_resize_seconds": seconds})
        for variant, probability in zip(VISUAL_VARIANTS, probabilities):
            row, sweep = evaluate(probability, target, {**base, "variant": variant,
                "prompt": f"support:{support['task_id']}", "support_task_id": str(support["task_id"]),
                "support_image_id": int(support["image_id"])})
            rows.append(row)
            sweep_rows.extend(sweep)
            save_mask(probability, variant, f"{task_id}__{support['task_id']}")
            if support_index == 0:
                first_probabilities[variant], first_ious[variant] = probability, row["iou"]
    if index < N_VISUALIZATIONS:
        save_pair_figure(image, target, pair, first_probabilities, first_ious, index)
    write_csv(OUT / "sample_metrics.csv", rows)
elapsed_all = time.perf_counter() - started_all
assert len(rows) == len(pairs) * (len(TEXT_VARIANTS) + len(VISUAL_VARIANTS) * SUPPORTS_PER_QUERY)
print(f"Completed {len(pairs)} queries × {len(VARIANTS)} variants; wall time {elapsed_all:.1f}s")


## 7. Bảng E2

`e2_variant_table.csv` là bảng chính. Cột `rows` là số dòng metric: N query với text,
N × K (query × support) với visual; mean IoU, cumulative IoU, AP và precision tính trên các dòng đó.
ΔIoU so với `text_phrase` tính **theo query**: IoU visual được trung bình trên K support trước khi trừ.
CI 95% bootstrap theo ảnh query. Visual prompt thường cần ngưỡng khác text, nên xem thêm cột
calibrated (ngưỡng chọn trên calibration) và AP (không phụ thuộc ngưỡng).


In [ ]:
image_of = {str(t["task_id"]): int(t["image_id"]) for t in selected}
reports, calibration_rows, table = {}, [], []
iou_by_variant = {}
for variant in VARIANTS:
    variant_rows = [r for r in rows if r["variant"] == variant]
    report, calibration_table = variant_report(variant_rows, [r for r in sweep_rows if r["variant"] == variant])
    reports[variant] = report
    calibration_rows += [{"variant": variant, **r} for r in calibration_table]
    per_query = defaultdict(list)
    for r in variant_rows:
        per_query[r["task_id"]].append(r["iou"])
    iou_by_variant[variant] = {task_id: float(np.mean(values)) for task_id, values in per_query.items()}
    reports[variant]["mean_iou_per_query"] = float(np.mean(list(iou_by_variant[variant].values())))
for variant in VARIANTS:
    comparison = None if variant == BASELINE_VARIANT else paired_comparison(
        iou_by_variant[BASELINE_VARIANT], iou_by_variant[variant], image_of, N_BOOTSTRAP, TIE_TOLERANCE, SEED)
    reports[variant]["paired_vs_baseline"] = comparison
    table.append(table_row(variant, reports[variant], comparison))
write_csv(OUT / "calibration_thresholds.csv", calibration_rows)
write_csv(OUT / "e2_variant_table.csv", table)
summary = {"experiment": "E2_text_vs_visual", "dataset": "VGPhraseCut", "split": "miniv",
    "n_queries": len(pairs), "supports_per_query": SUPPORTS_PER_QUERY, "baseline_variant": BASELINE_VARIANT,
    "variants": reports,
    "metric_note": ("Text rows: one per query; visual rows: one per query×support. "
                    "ΔIoU is per query after averaging visual IoU over supports; CI is image-level bootstrap.")}
write_json(OUT / "summary.json", summary)
write_json(OUT / "run.json", run_metadata({"experiment": "E2_text_vs_visual", "max_queries": MAX_QUERIES,
    "supports_per_query": SUPPORTS_PER_QUERY, "min_support_box_fraction": MIN_SUPPORT_BOX_FRACTION,
    "crop_padding": CROP_PADDING, "dark_factor": DARK_FACTOR, "variants": VARIANTS,
    "pairs_sha256": sha256(OUT / "e2_pairs.json"),
    "calibration_image_ids": sorted(calibration_ids),
    "evaluation_image_ids": sorted(set(query_image_ids) - calibration_ids)}))
print_table(table)


## 8. Xem hình và đóng gói

Tải `clipseg_e2_text_vs_visual.zip` trong Kaggle Output.


In [ ]:
for path in sorted((OUT / "visualizations").glob("*.png"))[:N_VISUALIZATIONS]:
    with Image.open(path) as image:
        display(image.copy())
archive = WORK_ROOT / f"{OUTPUT_NAME}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            package.write(path, arcname=path.relative_to(OUT).as_posix())
print("Result folder:", OUT)
print("Download:", archive)


## Nguồn và giới hạn

- [CLIPSeg / CVPR 2022](https://github.com/timojl/clipseg), [checkpoint HF](https://huggingface.co/CIDAS/clipseg-rd64-refined).
- [PhraseCutDataset](https://github.com/ChenyunWu/PhraseCutDataset): annotation, URL ảnh, split và quy ước polygon/mask.
- [CLIPSeg trong Transformers 4.55.4](https://huggingface.co/docs/transformers/v4.55.4/en/model_doc/clipseg).

Checkpoint pretrained, không train. PhraseCut liên quan tới dữ liệu huấn luyện CLIPSeg nên không gọi là zero-shot.
Google Drive/Visual Genome có thể gián đoạn; Add Input dataset chứa hai file JSON (và ảnh) là đường dự phòng.
